In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%cd /content/drive/MyDrive/constrained_KL_reg_Lin_bandit/

/content/drive/MyDrive/constrained_KL_reg_Lin_bandit


In [5]:
import numpy as np

class BoxTable:

    def __init__(self, means,is_one_hot=1, noise_std=0.0, seed=0):
        self.means = np.asarray(means, dtype=float)  # one mean per box/arm
        self.nA = self.means.shape[0]
        self.noise_std = float(noise_std)
        self.rng = np.random.default_rng(seed)
        self.is_one_hot = is_one_hot
        # self.neg_util = neg_util

    def __getitem__(self, a):
        """
        a is expected to be a one-hot vector (or close to it).
        We map it to an arm index via argmax.
        """
        a = np.asarray(a)
        if not self.is_one_hot:
          idx = int(np.argmax(a))  # box index
        else:
          idx = a
        noise = self.rng.normal(0.0, self.noise_std)
        #print("Inside getitem:",self.means[idx])
        return float(self.means[idx] + noise)


def make_tabular_box_environment(mu_r, mu_u,is_one_hot=0, neg_util=0,noise_r=0.0, noise_u=0.0, seed=0):
    """
    Returns (actions, R, U).

    actions: one-hot vectors (nA x nA), can sample ak as a vector.
    R: BoxTable that returns reward when indexed by ak
    U: BoxTable that returns utility when indexed by ak
    """
    mu_r = np.asarray(mu_r, dtype=float)
    mu_u = np.asarray(mu_u, dtype=float)
    assert mu_r.shape == mu_u.shape

    nA = mu_r.shape[0]
    actions = np.eye(nA)  # one-hot representation for each box

    R = BoxTable(mu_r,is_one_hot, noise_std=noise_r, seed=seed + 1)
    if neg_util==0:
      U = BoxTable(mu_u,is_one_hot, noise_std=noise_u, seed=seed + 2)
    else:
      U = BoxTable(-mu_u,is_one_hot, noise_std=noise_u, seed=seed + 2)

    return actions, R, U

In [6]:
## Unconstrained Lin-UCB

class LinUCB:
    def __init__(self, beta, rho, K, ndim, R=None,U=None, theta_star_r=None,theta_star_g=None, rad=1.0):
        self.beta = beta
        self.rho = rho
        self.K = K
        self.ndim = ndim
        self.theta_star_r = theta_star_r
        self.theta_star_g = theta_star_g
        self.rad = rad
        self.R = R
        self.U = U
        #print(self.beta,"\n",self.rho,'\n',self.K,'\n',self.ndim,'\n',self.theta_star,'\n',self.rad,'\n',self.R,'\n',self.U)
        #input()

        self.Lambda = rho * np.eye(ndim)
        self.b = np.zeros(ndim)

    def project(self, z):
        if np.linalg.norm(z) > self.rad:
            return z * self.rad / np.linalg.norm(z)
        return z

    def onehot(self, z):
        a = np.zeros(self.ndim)
        a[int(z)] = 1.0
        return a

    def Rew(self, a,rch=0):
      if rch==0:
        return self.R[np.argmax(a)]
      else:
        return np.dot(a, self.theta_star_r)

    def Util(self,a,uch=0):
      if uch==0:
        return self.U[np.argmax(a)]
      else:
        return np.dot(a,self.theta_star_g)

    def select_action(self, actions=None, ch=0):
        Lambda_inv = np.linalg.inv(self.Lambda)
        theta_hat = Lambda_inv @ self.b

        if ch == 0:  # discrete arms
            best_val = -1e18
            best_a = None
            for i in range(self.ndim):
                a = self.onehot(i)
                val = np.dot(a, theta_hat) + self.beta * np.sqrt(a.T @ Lambda_inv @ a)
                if val > best_val:
                    best_val = val
                    best_a = a
            return best_a

        else:  # continuous convex ball
            a = self.project(np.random.randn(self.ndim))
            for _ in range(200):
                bonus = np.sqrt(a @ Lambda_inv @ a) + 1e-9
                grad = theta_hat + self.beta * (Lambda_inv @ a) / bonus
                a = self.project(a + 0.05 * grad)
            return a

    def run_algo(self, ch=0,rch=0,uch=0):
        action_arr, rew_arr, util_arr = [], [],[]
        for _ in range(self.K):
            a = self.select_action(ch=ch)
            # a = self.select_action(ch=ch)
            r = self.Rew(a,rch)
            u = self.Util(a,uch)

            action_arr.append(int(np.argmax(a)))
            rew_arr.append(r)
            util_arr.append(u)

            self.Lambda += np.outer(a, a)
            self.b += a * r

        return action_arr, rew_arr,util_arr

In [ ]:
#Case 1a:-  3 arms

mu_r = [0.95, 0.70, 0.55]
mu_u = [0.20, 0.80, 0.95]

# constraint threshold b
b = 0.60

# Build environment objects compatible with R[ak], U[ak]
actions, R, U = make_tabular_box_environment(mu_r, mu_u, neg_util=0,noise_r=0.05, noise_u=0.05, seed=0)
nA = actions.shape[0]

beta_r=1.0
beta_g=1.0
lambda_=5.0
tau=1.0
rho=1.0
kappa=30.0
a=None
actions=actions
b=b
R=R
U=U
K=1000

agent = LinUCB(beta_r,rho,K,nA,R,U,None,1.0) #beta, rho, K, ndim, R=None,U=None, theta_star=None, rad=1.0
actions, rew_arr,util_arr = agent.run_algo(ch=0,rch=0,uch=0)

import pandas as pd
df_rew = pd.DataFrame(rew_arr,columns=['reward'])
df_util = pd.DataFrame(util_arr,columns=['utility'])
df_rew.to_excel('Case1_unconstrained_Lin_UCB_3_arms_reward.xlsx')
df_util.to_excel('Case1_unconstrained_Lin_UCB_3_arms_utility.xlsx')

import pickle
with open('Case1_unconstrained_Lin_UCB_3_arms_actions.pkl', 'wb') as f:
    pickle.dump(actions, f)
f.close()

In [ ]:
#For 10 arms
mu_r = [0.95, 0.90, 0.85, 0.80, 0.75, 0.70, 0.65, 0.60, 0.55, 0.50]
mu_u = [0.20, 0.30, 0.40, 0.55, 0.60, 0.70, 0.80, 0.90, 0.95, 0.99]

# constraint threshold b
b = 0.60

# Build environment objects compatible with R[ak], U[ak]
actions, R, U = make_tabular_box_environment(mu_r, mu_u,is_one_hot=1, neg_util=1,noise_r=0.05, noise_u=0.05, seed=0)
nA = actions.shape[0]

# Create  agent
beta_r=1.0
beta_g=1.0
lambda_=5.0
tau=3.0
rho=1.0
kappa=35.0
K=10000
eta = 0.1
is_tab = 1

# Create  agent
# beta_r=1.0
# beta_g=1.0
# lambda_=5.0
# tau=3.0
# rho=1.0
# kappa=35.0
# K=1000
# a=None
# nA=nA
# actions=actions
# b=b
# R=R
# U=U

agent = LinUCB(beta_r,rho,K,nA,R,U,None,1.0) #beta, rho, K, ndim, R=None,U=None, theta_star=None, rad=1.0
actions, rew_arr,util_arr = agent.run_algo(ch=0,rch=0,uch=0)

import pandas as pd
df_rew = pd.DataFrame(rew_arr,columns=['reward'])
df_util = pd.DataFrame(util_arr,columns=['utility'])
df_rew.to_excel('Case1_unconstrained_Lin_UCB_10_arms_reward.xlsx')
df_util.to_excel('Case1_unconstrained_Lin_UCB_10_arms_utility.xlsx')

import pickle
with open('Case1_unconstrained_Lin_UCB_10_arms_actions.pkl', 'wb') as f:
    pickle.dump(actions, f)
f.close()

Lets see the same CASE 1 but with constrained Lin-UCB

In [7]:
import numpy as np

class TabularPrimalDualLinUCB:
    def __init__(self, R, G, b,ndim,
                 beta_r=1.0, beta_g=1.0,
                 rho=1.0, eta=0.05, K=500):

        self.R = R   # reward table
        self.G = G   # cost table
        self.nA = ndim
        self.b = b

        self.beta_r = beta_r
        self.beta_g = beta_g
        self.rho = rho
        self.eta = eta
        self.K = K

        self.N = np.zeros(self.nA)   # action counts
        self.lambda_dual = 50.0

    def bonus(self, a):
        return 1.0 / np.sqrt(self.N[a] + self.rho)

    def select_action(self):
        values = []
        for a in range(self.nA):
            #print(a)
            val = (
                self.R[a]
                + self.beta_r * self.bonus(a)
                - self.lambda_dual * (
                    self.G[a]
                    + self.beta_g * self.bonus(a)
                    - self.b
                )
            )
            values.append(val)
        return int(np.argmax(values))

    def run_algo(self):
        actions, rewards, costs, lambdas = [], [], [], []

        for _ in range(self.K):
            a = self.select_action()
            #print(a)
            r = self.R[a]
            g = self.G[a]
            #print("Action selected:",a)
            #print("Cost returned:",g)
            #print("Raw output:",self.G[a])
            #input()

            actions.append(a)
            rewards.append(r)
            costs.append(g)
            lambdas.append(self.lambda_dual)

            self.N[a] += 1
            self.lambda_dual = max(
                0.0,
                self.lambda_dual + self.eta * (g - self.b)
            )

        return actions, rewards, costs, lambdas

In [8]:
class PrimalDualLinUCB:
    def __init__(self, beta_r, beta_g, rho, K, ndim, b,
                 theta_star_r, theta_star_g,
                 eta=0.05,lambda_dual=10):

        self.beta_r = beta_r
        self.beta_g = beta_g
        self.beta = beta_r
        self.rho = rho
        self.K = K
        self.ndim = ndim
        self.b = b
        self.eta = eta

        self.theta_star_r = theta_star_r
        self.theta_star_g = theta_star_g

        self.Lambda = rho * np.eye(ndim)
        self.b_r = theta_star_r.copy()#np.zeros(ndim)
        self.b_g = theta_star_g.copy()#np.zeros(ndim)
        self.counter_arm = np.zeros(ndim)

        self.lambda_dual = lambda_dual

    def onehot(self, i):
        a = np.zeros(self.ndim)
        a[i] = 1.0
        return a

    def Rew(self, a):
        return np.dot(a, self.theta_star_r)

    def Cost(self, a):
        return np.dot(a, self.theta_star_g)

    def select_action(self,t):
        Lambda_inv = np.linalg.inv(self.Lambda)
        theta_r = Lambda_inv @ self.b_r
        theta_g = Lambda_inv @ self.b_g

        best_val = -1e18
        best_a = None
        arm_chosen = None
        vals = []

        for i in range(self.ndim):
            a = self.onehot(i)
            #bonus = np.sqrt(np.log(t+1)/(self.counter_arm[i]+1))
            bonus = np.sqrt(a @ Lambda_inv @ a)
            #bonus *= np.sqrt(np.log(i + 1))
            # print(a)
            # print(theta_r)
            val = (
                a @ theta_r
                + self.beta * bonus
                - self.lambda_dual * (
                    a @ theta_g
                    - self.b
                )
            )
            # print("t and lambda_inv:",t," and ",Lambda_inv)
            # print(a,a@theta_r,a@theta_g)
            # print(bonus)
            # print(val)
            vals.append(val)
            if val > best_val:
                best_val = val
                best_a = a
                arm_chosen = i
        #input()
        self.counter_arm[arm_chosen]+=1
        return best_a,vals

    def run_algo(self):
        actions, rewards, costs, lambdas,step_vals = [], [], [], [],[]
        for _ in range(self.K):
            a,vals = self.select_action(_)

            r = self.Rew(a)
            g = self.Cost(a)
            step_vals.append(vals)

            actions.append(int(np.argmax(a)))
            rewards.append(r)
            costs.append(g)
            lambdas.append(self.lambda_dual)

            # primal update
            self.Lambda += np.outer(a, a)
            self.b_r += a * r
            self.b_g += a * (g)
            #print("b_r:",self.b_r," and b_g:",self.b_g)

            # dual update
            self.lambda_dual = max(
                0.0,
                self.lambda_dual + self.eta * (g - self.b)
            )

        return actions, rewards, costs, lambdas,step_vals

In [9]:
class Agent:
  def __init__(self,beta_r,beta_g,rho,K,ndim,b,R,U,theta_star_r,theta_star_g,is_tab=0,eta=0.1,lambda_dual=0):
    #Tabular:  R, G, b,beta_r=1.0, beta_g=1.0,rho=1.0, eta=0.05, K=500
    #Linear: beta_r, beta_g, rho, K, ndim, b,theta_star_r, theta_star_g,eta=0.05
    if is_tab==0:
      self.agent = PrimalDualLinUCB(beta_r,beta_g,rho,K,ndim,b,theta_star_r,theta_star_g,eta,lambda_dual)
    else:
      self.agent = TabularPrimalDualLinUCB(R,U,b,ndim,beta_r,beta_g,rho,eta,K)
  def run_algo(self):
    return self.agent.run_algo()

In [10]:
#Case 1a:-  3 arms
from collections import Counter

mu_r = [0.95, 0.70, 0.55]
mu_u = [0.20, 0.80, 0.95]

# constraint threshold b
b = 0.60

# Build environment objects compatible with R[ak], U[ak]
actions, R, U = make_tabular_box_environment(mu_r, mu_u,is_one_hot=1, neg_util=1,noise_r=0.05, noise_u=0.05, seed=0)
nA = actions.shape[0]

beta_r=1.0
beta_g=1.0
lambda_=50.0
tau=1.0
rho=1.0
kappa=30.0
a=None
actions=actions
U=U
K=10000
eta = 0.1
is_tab = 1

agent=Agent(beta_r,beta_g,rho,K,nA,-b,R,U,None,None,is_tab,eta) #beta_r,beta_g,rho,K,ndim,b,R,U,theta_star_r,theta_star_g,is_tab=0,eta=0.1
actions, rew_arr,util_arr,lambdas = agent.run_algo()

import pandas as pd
df_rew = pd.DataFrame(rew_arr,columns=['reward'])
df_util = pd.DataFrame(util_arr,columns=['utility'])
df_rew.to_excel('Case1_constrained_Lin_UCB_3_arms_reward.xlsx')
df_util.to_excel('Case1_constrained_Lin_UCB_3_arms_utility.xlsx')

import pickle
with open('Case1_constrained_Lin_UCB_3_arms_actions.pkl', 'wb') as f:
    pickle.dump(actions, f)
f.close()

with open('Case1_constrained_Lin_UCB_3_arms_lambdas.pkl', 'wb') as f:
    pickle.dump(lambdas, f)
f.close()
print(Counter(actions))

Counter({1: 4876, 0: 3027, 2: 2097})


In [11]:
#For 10 arms
from collections import Counter


mu_r = [0.95, 0.90, 0.85, 0.80, 0.75, 0.70, 0.65, 0.60, 0.55, 0.50]
mu_u = [0.20, 0.30, 0.40, 0.55, 0.60, 0.70, 0.80, 0.90, 0.95, 0.99]

# constraint threshold b
b = 0.60

# Build environment objects compatible with R[ak], U[ak]
actions, R, U = make_tabular_box_environment(mu_r, mu_u,is_one_hot=1, neg_util=1,noise_r=0.05, noise_u=0.05, seed=0)
nA = actions.shape[0]

# Create  agent
beta_r=1.0
beta_g=1.0
lambda_=0.0
tau=3.0
rho=1.0
kappa=35.0
K=10000
eta = 0.1
is_tab = 1

agent=Agent(beta_r,beta_g,rho,K,nA,-b,R,U,None,None,is_tab,eta) #beta_r,beta_g,rho,K,ndim,b,R,U,theta_star_r,theta_star_g,is_tab=0,eta=0.1
actions, rew_arr,util_arr,lambdas = agent.run_algo()

import pandas as pd
df_rew = pd.DataFrame(rew_arr,columns=['reward'])
df_util = pd.DataFrame(util_arr,columns=['utility'])
df_rew.to_excel('Case1_constrained_Lin_UCB_10_arms_reward.xlsx')
df_util.to_excel('Case1_constrained_Lin_UCB_10_arms_utility.xlsx')

import pickle
with open('Case1_constrained_Lin_UCB_10_arms_actions.pkl', 'wb') as f:
    pickle.dump(actions, f)
f.close()

with open('Case1_constrained_Lin_UCB_10_arms_lambdas.pkl', 'wb') as f:
    pickle.dump(lambdas, f)
f.close()
print(Counter(actions))

Counter({3: 1498, 9: 1386, 7: 1027, 6: 983, 5: 950, 4: 912, 0: 896, 2: 892, 1: 832, 8: 624})


#CASE 2: Linearized reward and utility given already known $\theta^{*}_{r}$ and $\theta_{g}^{*}$

In [ ]:
#Case 2, subcase a: finite arms and theta_star reward and utility
import numpy as np
import pandas as pd
from collections import Counter

def make_case2_toy_env(ndim=10, seed=0,is_neg_util=0):
    """
    Case 2 toy env (NO noise), one-hot features:
      - arms: i = 0..ndim-1
      - action vector a = onehot(i) in R^{ndim}
      - reward  r(a) = <a, theta_star_r>
      - utility g(a) = <a, theta_star_u>
    """
    rng = np.random.default_rng(seed)

    # "actions" list: indices; update_policy will convert each index to one-hot
    actions = list(range(ndim))

    # True parameters (fixed, deterministic)
    # Make one arm clearly best for reward, and some arms better for utility.
    theta_star_r = np.linspace(0.1, 1.0, ndim)          # increasing reward
    if is_neg_util:
      theta_star_u = -np.linspace(1.0, 0.1, ndim)          # decreasing utility (tradeoff)
    else:
      theta_star_u = np.linspace(1.0, 0.1, ndim)          # decreasing utility (tradeoff)

    # Constraint threshold on expected utility (tuneable)
    b = 0.7

    return actions, theta_star_r, theta_star_u, b


# quick sanity check
ndim = 10
is_neg_util = 1
actions, theta_star_r, theta_star_g, b = make_case2_toy_env(ndim=ndim, seed=0,is_neg_util=is_neg_util)
print("ndim =", len(actions))
print("theta_star_r =", np.round(theta_star_r, 3))
print("theta_star_u =", np.round(theta_star_g, 3))
print("b =", b)

#actions, theta_star_r, theta_star_g, b = make_case2_toy_env(ndim=ndim, seed=0)

'''agent = ConstrainedLinUCB(
    beta_r=0.01,#decrease maybe
    beta_g=0.01,#decrease maybe
    #lambda_=50.0,#can increase
    #tau=0.3,#little less can increase >1
    rho=0.3, # decrease and keep <1 keep 0.4 or 0.3 maybe
    #kappa=2.0,#decrease this too high
    K=1000,
    ndim=len(actions),
    b=b,
    actions=actions,
    theta_star_r=theta_star_r,
    theta_star_u=theta_star_u
)  #beta_r, beta_g, rho, K, ndim, b, R=None,U=None,theta_star_r=None, theta_star_g=None, rad=1.0'''

##beta_r,beta_g,rho,K,ndim,b,R,U,theta_star_r,theta_star_g,is_tab=0,eta=0.1
beta_r = 1
beta_g = 1
rho = 0.3
K = 10000
R = None
U = None
is_tab=0
eta = 0.1

agent = Agent(beta_r,beta_g,rho,K,ndim,-b,R,U,theta_star_r,theta_star_g,is_tab=is_tab,eta=eta)
actions,rew_arr,util_arr,lambdas,step_vals = agent.run_algo()

import pandas as pd
df_rew = pd.DataFrame(rew_arr,columns=['reward'])
df_util = pd.DataFrame(util_arr,columns=['utility'])
df_rew.to_excel('Case2_constrained_Lin_UCB_10_arms_reward.xlsx')
df_util.to_excel('Case2_constrained_Lin_UCB_10_arms_utility.xlsx')

import pickle
with open('Case2_constrained_Lin_UCB_10_arms_actions.pkl', 'wb') as f:
    pickle.dump(actions, f)
f.close()

with open('Case2_constrained_Lin_UCB_10_arms_lambdas.pkl', 'wb') as f:
    pickle.dump(lambdas, f)
f.close()
print(Counter(actions))

ndim = 10
theta_star_r = [0.1 0.2 0.3 0.4 0.5 0.6 0.7 0.8 0.9 1. ]
theta_star_u = [-1.  -0.9 -0.8 -0.7 -0.6 -0.5 -0.4 -0.3 -0.2 -0.1]
b = 0.7
Counter({0: 3103, 1: 1746, 2: 1211, 3: 927, 4: 745, 5: 612, 6: 513, 7: 436, 8: 375, 9: 332})


#Unconstrained Case 2

In [ ]:
import numpy as np
import pandas as pd
from collections import Counter

def make_case2_toy_env(ndim=10, seed=0,is_neg_util=0):
    """
    Case 2 toy env (NO noise), one-hot features:
      - arms: i = 0..ndim-1
      - action vector a = onehot(i) in R^{ndim}
      - reward  r(a) = <a, theta_star_r>
      - utility g(a) = <a, theta_star_u>
    """
    rng = np.random.default_rng(seed)

    # "actions" list: indices; update_policy will convert each index to one-hot
    actions = list(range(ndim))

    # True parameters (fixed, deterministic)
    # Make one arm clearly best for reward, and some arms better for utility.
    theta_star_r = np.linspace(0.1, 1.0, ndim)          # increasing reward
    if is_neg_util:
      theta_star_u = -np.linspace(1.0, 0.1, ndim)          # decreasing utility (tradeoff)
    else:
      theta_star_u = np.linspace(1.0, 0.1, ndim)          # decreasing utility (tradeoff)

    # Constraint threshold on expected utility (tuneable)
    b = 0.7

    return actions, theta_star_r, theta_star_u, b


# quick sanity check
ndim = 10
is_neg_util = 0
actions, theta_star_r, theta_star_g, b = make_case2_toy_env(ndim=ndim, seed=0,is_neg_util=is_neg_util)
print("ndim =", len(actions))
print("theta_star_r =", np.round(theta_star_r, 3))
print("theta_star_u =", np.round(theta_star_g, 3))
print("b =", b)

beta_r = 1
beta_g = 1
rho = 0.3
K = 10000
R = None
U = None
is_tab=0
eta = 0.1

agent = LinUCB(beta_r,rho,K,ndim,R,U,theta_star_r,theta_star_g,1.0) #beta, rho, K, ndim, R=None,U=None, theta_star_r=None,theta_star_g=None, rad=1.0
actions, rew_arr,util_arr = agent.run_algo(ch=0,rch=1,uch=1)

import pandas as pd
df_rew = pd.DataFrame(rew_arr,columns=['reward'])
df_util = pd.DataFrame(util_arr,columns=['utility'])
df_rew.to_excel('Case2_unconstrained_Lin_UCB_10_arms_reward.xlsx')
df_util.to_excel('Case2_unconstrained_Lin_UCB_10_arms_utility.xlsx')

import pickle
with open('Case2_unconstrained_Lin_UCB_10_arms_actions.pkl', 'wb') as f:
    pickle.dump(actions, f)
f.close()
print(Counter(actions))

ndim = 10
theta_star_r = [0.1 0.2 0.3 0.4 0.5 0.6 0.7 0.8 0.9 1. ]
theta_star_u = [1.  0.9 0.8 0.7 0.6 0.5 0.4 0.3 0.2 0.1]
b = 0.7
Counter({9: 9877, 8: 78, 7: 21, 6: 9, 5: 5, 4: 3, 1: 2, 2: 2, 3: 2, 0: 1})
